# Deployment & Model Export

Prototype deployment with **live character streaming** (one letter at a time). No full app yet.

**Prereqs:** notebooks 2 & 5. Optional: `pip install fastapi`.

Run top to bottom: load model → stream engine → dialogue → export → four app prototypes.


## 1. Setup

Paths and `DEPLOY_CONFIG`. Set `stream_delay_s=0` for fast typewriter demos.


In [ ]:
from pathlib import Path
import json
import math
import copy
import re
import time
import textwrap
from dataclasses import dataclass, field
from typing import Iterator

import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    from fastapi import FastAPI
    from fastapi.responses import StreamingResponse
    from fastapi.testclient import TestClient
    HAS_FASTAPI = True
except ImportError:
    HAS_FASTAPI = False
    print("fastapi not installed — skip section 5 or run: pip install fastapi")


In [ ]:
cwd = Path.cwd()
if (cwd / "data").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError(f"Could not locate 'data' from cwd: {cwd}")

DATA_DIR = PROJECT_ROOT / "data"
ARTIFACTS_DIR = DATA_DIR / "artifacts"
CHECKPOINT_DIR = ARTIFACTS_DIR / "model"
DEPLOY_DIR = ARTIFACTS_DIR / "deploy"

LOCAL_CKPT = CHECKPOINT_DIR / "gpt_best.pt"
VOCAB_PATH = ARTIFACTS_DIR / "char_vocab.json"

DEPLOY_CONFIG = {
    "seed": 42,
    "stream_delay_s": 0.015,  # typewriter pause between chars; use 0 for fast runs
    "default_temperature": 0.8,
    "default_top_k": 40,
    "default_top_p": 0.9,
}

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Project root: {PROJECT_ROOT}")
print(f"Device: {device}")


### 1.1 Vocabulary

65-char mapping from notebook 2 (`encode` / `decode`).


In [ ]:
assert VOCAB_PATH.exists(), f"Missing {VOCAB_PATH}. Run preprocessing first."

with open(VOCAB_PATH, "r", encoding="utf-8") as f:
    vocab_payload = json.load(f)

vocab_size = vocab_payload["vocab_size"]
stoi = vocab_payload["stoi"]
itos = {int(k): v for k, v in vocab_payload["itos"].items()}


def encode(text: str) -> list[int]:
    return [stoi[ch] for ch in text if ch in stoi]


def decode(indices) -> str:
    return "".join(itos[int(i)] for i in indices)


print(f"Vocabulary size: {vocab_size}")


### 1.2 Model

Same GPT as notebooks 5–7. Three cells: attention → block → full model.


In [ ]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dropout: float):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.qkv_proj = nn.Linear(d_model, 3 * d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        self.register_buffer("mask", None, persistent=False)

    def _causal_mask(self, size: int, device: torch.device):
        if self.mask is None or self.mask.size(0) < size:
            self.mask = torch.tril(torch.ones(size, size, device=device)).view(1, 1, size, size)
        return self.mask[:, :, :size, :size]

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.size()
        qkv = self.qkv_proj(x)
        q, k, v = qkv.chunk(3, dim=-1)

        def reshape_heads(t):
            return t.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        q, k, v = reshape_heads(q), reshape_heads(k), reshape_heads(v)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        mask = self._causal_mask(T, x.device)
        att = att.masked_fill(mask[:, :, :T, :T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)
        att = self.attn_dropout(att)
        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(y))


In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, d_ff_mult: int, dropout: float):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.attn = MultiHeadSelfAttention(d_model, n_heads, dropout)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff_mult * d_model),
            nn.GELU(),
            nn.Linear(d_ff_mult * d_model, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x


In [ ]:
class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size: int, config: dict):
        super().__init__()
        d_model = config["d_model"]
        n_heads = config["n_heads"]
        n_layers = config["n_layers"]
        dropout = config["dropout"]
        block_size = config["block_size"]
        d_ff_mult = config.get("d_ff", 4)
        self.block_size = block_size
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.pos_embedding = nn.Embedding(block_size, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList(
            [TransformerBlock(d_model, n_heads, d_ff_mult, dropout) for _ in range(n_layers)]
        )
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, idx: torch.Tensor, targets: torch.Tensor | None = None):
        B, T = idx.size()
        if T > self.block_size:
            raise ValueError(f"Sequence length {T} exceeds block_size {self.block_size}")
        pos = torch.arange(0, T, device=idx.device, dtype=torch.long).unsqueeze(0)
        x = self.drop(self.token_embedding(idx) + self.pos_embedding(pos))
        for block in self.blocks:
            x = block(x)
        logits = self.head(self.ln_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(B * T, -1), targets.view(B * T))
        return logits, loss


### 1.3 Checkpoint

Load `gpt_best.pt`, `model.eval()`.


In [ ]:
def _load_checkpoint_file(ckpt_path: Path):
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    if isinstance(ckpt, dict) and "model" in ckpt and "config" in ckpt:
        return ckpt["model"], copy.deepcopy(ckpt["config"])
    raise ValueError(f"Unexpected checkpoint format at {ckpt_path}")


assert LOCAL_CKPT.exists(), f"Missing {LOCAL_CKPT}. Run notebook 5 first."
state_dict, model_config = _load_checkpoint_file(LOCAL_CKPT)

model = GPTLanguageModel(vocab_size=vocab_size, config=model_config).to(device)
model.load_state_dict(state_dict)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"Loaded {LOCAL_CKPT.name}")
print(f"Parameters: {n_params:,}")
print(f"Architecture: {model_config['n_layers']} layers, d_model={model_config['d_model']}, block_size={model_config['block_size']}")


## 2. Live streaming

One char at a time. Stack: `sample_next_token` → `stream_chars` → `live_print`.


In [ ]:
CHAR_HEADER_RE = re.compile(r"^[A-Z][A-Z\s]+:")


def set_seed(seed: int | None) -> None:
    if seed is None:
        return
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def _apply_sampling(logits: torch.Tensor, temperature: float, top_k: int, top_p: float) -> torch.Tensor:
    # Temperature scales randomness; top-k / top-p trim unlikely tokens
    logits = logits / max(temperature, 1e-8)
    if top_k > 0:
        k = min(top_k, logits.size(-1))
        v, _ = torch.topk(logits, k, dim=-1)
        logits = logits.masked_fill(logits < v[:, [-1]], float("-inf"))
    if top_p < 1.0:
        sorted_logits, sorted_idx = torch.sort(logits, descending=True, dim=-1)
        probs = F.softmax(sorted_logits, dim=-1)
        cum_probs = torch.cumsum(probs, dim=-1)
        remove = cum_probs - probs >= top_p
        sorted_logits = sorted_logits.masked_fill(remove, float("-inf"))
        logits = sorted_logits.scatter(1, sorted_idx, sorted_logits)
    return logits


### 2.2 Next token

One forward pass → one sampled character.


In [ ]:
@torch.no_grad()
def sample_next_token(
    context: torch.Tensor,
    *,
    temperature: float = 0.8,
    top_k: int = 40,
    top_p: float = 0.9,
) -> tuple[torch.Tensor, str]:
    idx_cond = context[:, -model.block_size :]
    logits, _ = model(idx_cond)
    logits = _apply_sampling(logits[:, -1, :], temperature, top_k, top_p)
    probs = F.softmax(logits, dim=-1)
    next_id = torch.multinomial(probs, num_samples=1)
    ch = decode(next_id[0].tolist())
    return torch.cat([context, next_id], dim=1), ch


### 2.3 `stream_chars`

Main generator. `stop_on_double_newline=True` stops after a blank line.


In [ ]:
@torch.no_grad()
def stream_chars(
    prompt: str,
    max_new_tokens: int = 200,
    *,
    temperature: float | None = None,
    top_k: int | None = None,
    top_p: float | None = None,
    seed: int | None = DEPLOY_CONFIG["seed"],
    stop_on_double_newline: bool = False,
) -> Iterator[str]:
    temperature = DEPLOY_CONFIG["default_temperature"] if temperature is None else temperature
    top_k = DEPLOY_CONFIG["default_top_k"] if top_k is None else top_k
    top_p = DEPLOY_CONFIG["default_top_p"] if top_p is None else top_p

    set_seed(seed)
    ids = encode(prompt)
    if not ids:
        raise ValueError("Prompt has no valid vocabulary characters.")

    context = torch.tensor([ids], dtype=torch.long, device=device)
    emitted = 0
    trailing_newlines = 0

    for _ in range(max_new_tokens):
        context, ch = sample_next_token(context, temperature=temperature, top_k=top_k, top_p=top_p)
        yield ch
        emitted += 1
        if stop_on_double_newline:
            trailing_newlines = trailing_newlines + 1 if ch == chr(10) else 0
            if trailing_newlines >= 2 and emitted > 8:
                break


In [ ]:
def live_print(char_iter: Iterator[str], *, delay_s: float | None = None, prefix: str = "") -> str:
    delay_s = DEPLOY_CONFIG["stream_delay_s"] if delay_s is None else delay_s
    if prefix:
        print(prefix, end="", flush=True)
    text = prefix
    for ch in char_iter:
        print(ch, end="", flush=True)
        text += ch
        if delay_s:
            time.sleep(delay_s)
    print()
    return text


### 2.5 Demo

Output types out letter by letter. Look for `NAME:` headers and dialogue.


In [ ]:
print("Streaming (one character at a time):\n")
live_text = live_print(
    stream_chars("ROMEO:", max_new_tokens=180, seed=42, stop_on_double_newline=True),
    delay_s=0.01,
    prefix="ROMEO:",
)
new_chars = len(live_text) - len("ROMEO:")
print(f"\nInference note: generated {new_chars} new characters after the prompt.")


**Inference:** Play formatting = model learned structure. Gibberish → lower `temperature`.


## 3. Character dialogue

`CharacterSession` — turns + growing script context.


In [ ]:
@dataclass
class CharacterSession:
    character: str
    context: str = field(default="", init=False)
    history: list[dict] = field(default_factory=list)

    def __post_init__(self):
        self.character = self.character.strip().upper()
        self.context = f"{self.character}:"

    def stream_speech(self, max_new_tokens: int = 160, *, seed: int | None = None, **kwargs) -> Iterator[str]:
        spoken_parts = []
        for ch in stream_chars(self.context, max_new_tokens=max_new_tokens, seed=seed, **kwargs):
            spoken_parts.append(ch)
            yield ch
        spoken = "".join(spoken_parts)
        self.context += spoken
        self.history.append({"role": self.character, "text": spoken})

    def add_cue(self, speaker: str, line: str) -> None:
        cue = f"\n\n{speaker.strip().upper()}:\n{line.strip()}\n\n"
        self.context += cue
        self.history.append({"role": speaker.strip().upper(), "text": line.strip()})

    def reply_stream(self, other_speaker: str, line: str, **kwargs) -> Iterator[str]:
        self.add_cue(other_speaker, line)
        return self.stream_speech(**kwargs)


### 3.2 Demo

ROMEO speaks, Juliet cues, ROMEO replies with context.


In [ ]:
session = CharacterSession("ROMEO")

print("Turn 1 — ROMEO speaks:")
live_print(session.stream_speech(seed=7), prefix=f"{session.character}:")


In [ ]:
session.add_cue("JULIET", "O Romeo, what news from Verona?")

print("\nTurn 2 — ROMEO replies:")
live_print(session.reply_stream("JULIET", "O Romeo, what news from Verona?", seed=8))


In [ ]:
print("\nTranscript (inference audit):")
for turn in session.history:
    preview = turn["text"][:100].replace("\n", " ")
    print(f"  [{turn['role']}] {preview}{'...' if len(turn['text']) > 100 else ''}")


**Inference:** Turn 2 should echo the cue. Unrelated reply → context exceeded `block_size`.


## 4. Export bundle

`manifest.json` + `inference_engine.py` in `data/artifacts/deploy/`.


In [ ]:
DEPLOY_DIR.mkdir(parents=True, exist_ok=True)

manifest = {
    "version": "1.0.0",
    "model_checkpoint": str(LOCAL_CKPT.relative_to(PROJECT_ROOT)),
    "vocab_path": str(VOCAB_PATH.relative_to(PROJECT_ROOT)),
    "vocab_size": vocab_size,
    "model_config": model_config,
    "sampling_defaults": {
        "temperature": DEPLOY_CONFIG["default_temperature"],
        "top_k": DEPLOY_CONFIG["default_top_k"],
        "top_p": DEPLOY_CONFIG["default_top_p"],
    },
    "characters": ["ROMEO", "JULIET", "KING", "NURSE", "GLOUCESTER"],
}

manifest_path = DEPLOY_DIR / "manifest.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print(f"Wrote {manifest_path}")


### 4.2 Inference engine

Standalone module (duplicate classes for portability).


In [ ]:
engine_path = DEPLOY_DIR / "inference_engine.py"

ENGINE_TEMPLATE = r'''
# Minimal streaming inference engine — exported from notebook 8.
from __future__ import annotations

import json
import math
from pathlib import Path
from typing import Iterator

import torch
import torch.nn as nn
import torch.nn.functional as F


class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.qkv_proj = nn.Linear(d_model, 3 * d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        self.register_buffer("mask", None, persistent=False)

    def _causal_mask(self, size, device):
        if self.mask is None or self.mask.size(0) < size:
            self.mask = torch.tril(torch.ones(size, size, device=device)).view(1, 1, size, size)
        return self.mask[:, :, :size, :size]

    def forward(self, x):
        B, T, C = x.size()
        qkv = self.qkv_proj(x)
        q, k, v = qkv.chunk(3, dim=-1)
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        mask = self._causal_mask(T, x.device)
        att = att.masked_fill(mask[:, :, :T, :T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)
        att = self.attn_dropout(att)
        y = (att @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(y))


class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff_mult, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.attn = MultiHeadSelfAttention(d_model, n_heads, dropout)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff_mult * d_model), nn.GELU(),
            nn.Linear(d_ff_mult * d_model, d_model), nn.Dropout(dropout),
        )

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        return x + self.ff(self.ln2(x))


class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size, config):
        super().__init__()
        d_model = config["d_model"]
        self.block_size = config["block_size"]
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.pos_embedding = nn.Embedding(config["block_size"], d_model)
        self.drop = nn.Dropout(config["dropout"])
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, config["n_heads"], config.get("d_ff", 4), config["dropout"])
            for _ in range(config["n_layers"])
        ])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, idx, targets=None):
        B, T = idx.size()
        pos = torch.arange(0, T, device=idx.device, dtype=torch.long).unsqueeze(0)
        x = self.drop(self.token_embedding(idx) + self.pos_embedding(pos))
        for block in self.blocks:
            x = block(x)
        return self.head(self.ln_f(x)), None


class StreamingEngine:
    def __init__(self, project_root: Path):
        self.project_root = Path(project_root)
        manifest_file = self.project_root / "data/artifacts/deploy/manifest.json"
        with open(manifest_file, encoding="utf-8") as f:
            self.manifest = json.load(f)
        with open(self.project_root / self.manifest["vocab_path"], encoding="utf-8") as f:
            vocab = json.load(f)
        self.stoi = vocab["stoi"]
        self.itos = {int(k): v for k, v in vocab["itos"].items()}
        ckpt_path = self.project_root / self.manifest["model_checkpoint"]
        ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model = GPTLanguageModel(vocab["vocab_size"], ckpt["config"]).to(self.device)
        self.model.load_state_dict(ckpt["model"])
        self.model.eval()

    @torch.no_grad()
    def stream(self, prompt: str, max_new_tokens: int = 100, temperature: float = 0.8) -> Iterator[str]:
        ids = [self.stoi[c] for c in prompt if c in self.stoi]
        ctx = torch.tensor([ids], dtype=torch.long, device=self.device)
        for _ in range(max_new_tokens):
            ctx = ctx[:, -self.model.block_size :]
            logits, _ = self.model(ctx)
            logits = logits[:, -1, :] / max(temperature, 1e-8)
            probs = F.softmax(logits, dim=-1)
            nxt = torch.multinomial(probs, 1)
            ctx = torch.cat([ctx, nxt], dim=1)
            yield self.itos[int(nxt.item())]
'''.strip() + chr(10)

engine_path.write_text(ENGINE_TEMPLATE, encoding="utf-8")
print(f"Wrote {engine_path}")


In [ ]:
import importlib.util

spec = importlib.util.spec_from_file_location("inference_engine", engine_path)
engine_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(engine_mod)

exported = engine_mod.StreamingEngine(PROJECT_ROOT)
print("Exported StreamingEngine loaded successfully.")


In [ ]:
print("\nStreaming via exported module:")
print("JULIET:", end="", flush=True)
for i, ch in enumerate(exported.stream("JULIET:", max_new_tokens=80, temperature=0.8)):
    print(ch, end="", flush=True)
    if i >= 79:
        break
print("\n\nInference: if this matches notebook streaming, the bundle is portable.")


## 5. FastAPI prototype

`GET /stream` via `TestClient` — no server.


In [ ]:
if not HAS_FASTAPI:
    print("Install fastapi to run this section.")
else:
    app = FastAPI(title="Shakespeare GPT (prototype)")

    @app.get("/health")
    def health():
        return {"status": "ok", "device": device, "params": n_params}

    @app.get("/stream")
    def stream_endpoint(
        prompt: str = "ROMEO:",
        max_tokens: int = 120,
        temperature: float = 0.8,
        top_k: int = 40,
        top_p: float = 0.9,
    ):
        def body():
            for ch in stream_chars(
                prompt,
                max_new_tokens=max_tokens,
                temperature=temperature,
                top_k=top_k,
                top_p=top_p,
                seed=None,
            ):
                yield ch

        return StreamingResponse(body(), media_type="text/plain; charset=utf-8")

    print("Routes defined: GET /health, GET /stream")


In [ ]:
if HAS_FASTAPI:
    client = TestClient(app)
    print("Health check:", client.get("/health").json())

    with client.stream("GET", "/stream", params={"prompt": "KING:", "max_tokens": 90}) as resp:
        assert resp.status_code == 200
        streamed = "".join(resp.iter_text())

    print(f"Streamed {len(streamed)} characters over HTTP")
    print("Preview:", streamed[:200] + ("..." if len(streamed) > 200 else ""))
    print("\nInference: chunked plain text is enough for EventSource or fetch streaming in a browser.")


**Next:** `uvicorn` + browser streaming on `/stream`.


## 6. Stage improv

Three speakers, streamed lines, shared context.


In [ ]:
STAGE_CAST = ["ROMEO", "JULIET", "NURSE"]
stage_context = ""

print("Stage improv — three streamed turns:\n")
for idx, speaker in enumerate(STAGE_CAST):
    stage_context = f"{stage_context}\n\n{speaker}:" if stage_context else f"{speaker}:"
    print(f"[{speaker}]", end=" ", flush=True)

    new_chars = []
    for ch in stream_chars(
        stage_context,
        max_new_tokens=100,
        seed=(idx + 1) * 17,
        stop_on_double_newline=True,
    ):
        print(ch, end="", flush=True)
        new_chars.append(ch)
        time.sleep(0.005)

    print()
    stage_context += "".join(new_chars)


**Inference:** Later speakers may reference earlier ones; 128-token window limits coherence.


## 7. CLI prototype

Scripted cues → streamed replies.


In [ ]:
def cli_character_loop(character: str = "ROMEO", user_cues: list[str] | None = None, *, delay_s: float = 0.008):
    user_cues = user_cues or [
        "What dost thou say of the king?",
        "And what of Juliet?",
    ]
    session = CharacterSession(character)
    transcript = []

    for cue in user_cues:
        print(f"\nUSER> {cue}")
        print(f"{session.character}> ", end="", flush=True)
        reply = live_print(session.reply_stream("USER", cue, seed=len(transcript) + 1), delay_s=delay_s)
        transcript.append({"user": cue, "character": reply})

    return transcript


In [ ]:
cli_transcript = cli_character_loop("GLOUCESTER")
print(f"\nCompleted {len(cli_transcript)} simulated CLI turns.")


**Next:** use `input()` + exported `StreamingEngine`.


## 8. UI state

`LiveUIState` — buffer a chat UI would re-render per token.


In [ ]:
class LiveUIState:
    # Holds everything a simple chat UI would re-render after each token.

    def __init__(self, character: str = "ROMEO"):
        self.character = character.upper()
        self.display_text = f"{self.character}:"
        self.session = CharacterSession(self.character)

    def stream_turn(self, max_tokens: int = 100, seed: int | None = None) -> str:
        chunks = []
        for ch in self.session.stream_speech(max_new_tokens=max_tokens, seed=seed):
            chunks.append(ch)
            self.display_text += ch  # UI would call st.write(display_text) here
        return "".join(chunks)

    def user_interrupt(self, speaker: str, line: str, seed: int | None = None) -> str:
        chunks = []
        for ch in self.session.reply_stream(speaker, line, seed=seed):
            chunks.append(ch)
            self.display_text += ch
        return "".join(chunks)


In [ ]:
ui = LiveUIState("JULIET")
ui.stream_turn(seed=11)
ui.user_interrupt("ROMEO", "Fair Juliet, good morrow.")
ui.stream_turn(seed=12)

print("Last 400 characters of UI buffer:")
print(ui.display_text[-400:])
print(f"\nTotal buffer length: {len(ui.display_text)} chars")


**Inference:** Append each char to `display_text`; bind to a text widget.


## 9. Wrap-up

Validated streaming, dialogue, improv, CLI, export, FastAPI. Not built: Gradio/Streamlit/server. Artifacts in `data/artifacts/deploy/`.
